<a href="https://colab.research.google.com/github/Shineii86/LeechBot/blob/main/notebooks/LeechBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# @title ♻️ Google Drive Auth
#@markdown <br><center><img src='https://user-images.githubusercontent.com/125879861/255377947-6ac19c35-dbbd-4a9b-bc0e-c603de81c533.png' height="70" alt="Gdrive-logo"/></center>
#@markdown <center><h4><font color=orange><b>Secure Google Drive Integration</b></font></h4></center><br>

from google.colab import auth, drive
from google.auth.transport.requests import Request
import google.auth
import pickle
import os
import time
from IPython.display import clear_output, display, Markdown
import ipywidgets as widgets

# ─────────────────────────────────────────────────────────────
# 🎛️ Configuration Widget
# ─────────────────────────────────────────────────────────────
MODE = widgets.Dropdown(
    options=['Mount Drive', 'Unmount Drive', 'Generate Token', 'Nothing'],
    value='Mount Drive',
    description='🔧 Action:',
    style={'description_width': 'initial'}
)

MOUNT_PATH = widgets.Text(
    value='/content/drive',
    description='📁 Mount Path:',
    disabled=False,
    style={'description_width': 'initial'}
)

TOKEN_PATH = widgets.Text(
    value='/content/token.pickle',
    description='🔑 Token Save Path:',
    disabled=False,
    style={'description_width': 'initial'}
)

USE_COLAB_SECRETS = widgets.Checkbox(
    value=True,
    description='🔐 Use Colab Secrets for credentials (recommended)',
    indent=False
)

config_box = widgets.VBox([MODE, MOUNT_PATH, TOKEN_PATH, USE_COLAB_SECRETS])
display(config_box)

# ─────────────────────────────────────────────────────────────
# 🛠️ Helper Functions
# ─────────────────────────────────────────────────────────────
def print_status(emoji: str, message: str, level: str = "info"):
    """Unified status printer with color coding."""
    colors = {"info": "cyan", "success": "green", "error": "red", "warning": "orange"}
    color = colors.get(level, "white")
    display(Markdown(f"<font color={color}>[{emoji}] {message}</font>"))

def mount_drive(path: str = "/content/drive", force: bool = True):
    """Mount Google Drive with retry logic."""
    for attempt in range(3):
        try:
            print_status("🔗", f"Mounting Drive to {path}... (attempt {attempt+1}/3)")
            drive.mount(path, force_remount=force)
            print_status("✅", "Google Drive mounted successfully!", "success")
            return True
        except Exception as e:
            print_status("⚠️", f"Mount attempt {attempt+1} failed: {e}", "warning")
            time.sleep(2)
    print_status("❌", "Failed to mount Drive after 3 attempts", "error")
    return False

def unmount_drive():
    """Safely unmount Google Drive."""
    try:
        drive.flush_and_unmount()
        print_status("🔓", "Google Drive unmounted successfully!", "success")
        return True
    except Exception as e:
        print_status("⚠️", f"Unmount warning: {e}", "warning")
        return False

def generate_token(save_path: str = "/content/token.pickle"):
    """Generate & save token using Colab authentication."""
    try:
        # Authenticate user
        auth.authenticate_user()
        print_status("🔐", "User authenticated via Colab", "success")

        # Get credentials using google.auth.default()
        credentials, project = google.auth.default()

        # Refresh if needed
        if credentials.expired and credentials.refresh_token:
            credentials.refresh(Request())
            print_status("🔄", "Credentials refreshed", "info")

        # Save to pickle file
        with open(save_path, 'wb') as f:
            pickle.dump(credentials, f)

        # Set restrictive permissions
        os.chmod(save_path, 0o600)

        print_status("💾", f"Token saved securely to {save_path}", "success")
        return True

    except Exception as e:
        print_status("❌", f"Token generation failed: {e}", "error")
        return False

# ─────────────────────────────────────────────────────────────
# 🚀 Main Execution
# ─────────────────────────────────────────────────────────────
def execute():
    clear_output(wait=True)
    display(Markdown("### ⚙️ Processing Request...\n"))

    action = MODE.value
    mount_to = MOUNT_PATH.value
    token_to = TOKEN_PATH.value
    use_secrets = USE_COLAB_SECRETS.value

    # Handle Mount/Unmount
    if action == "Mount Drive":
        mount_drive(mount_to)
    elif action == "Unmount Drive":
        unmount_drive()

    # Handle Token Generation
    if action == "Generate Token":
        generate_token(token_to)

    # Combined operations (mount + token)
    if action == "Mount Drive" and use_secrets:
        generate_token(token_to)

    display(Markdown("\n---\n### ✅ Operation Complete!\n<sub>Tip: Store tokens in Colab Secrets for production use.</sub>"))

# Run when widget changes or manually
execute_button = widgets.Button(description="▶️ Execute", button_style="success", icon="check")
execute_button.on_click(lambda b: execute())
display(execute_button)

# Optional: auto-execute on first load
# execute()

In [ ]:

# =============================================================================
#  LeechBot - Advanced Telegram File Transloader
#  Copyright (c) 2026 Shinei Nouzen | GitHub: https://github.com/Shineii86
# =============================================================================
# @title **🚀 LeechBot Colab Deployer**
#@markdown <div align="center">
#@markdown   <img src="https://user-images.githubusercontent.com/125879861/255391401-371f3a64-732d-4954-ac0f-4f093a6605e1.png" width="600px">
#@markdown </div>
#@markdown
#@markdown **✨ Features**: Secrets Support • Auto-Recovery • GPU Optimization • Health Checks
#@markdown
#@markdown ---
#@markdown ## 🔐 **Credentials**
#@markdown
#@markdown | Field | Secret Key Name | Required |
#@markdown |-------|----------------|----------|
#@markdown | API_ID | `LEECHBOT_API_ID` | ✅ |
#@markdown | API_HASH | `LEECHBOT_API_HASH` | ✅ |
#@markdown | BOT_TOKEN | `LEECHBOT_BOT_TOKEN` | ✅ |
#@markdown | USER_ID | `LEECHBOT_USER_ID` | ✅ |
#@markdown | DUMP_ID | `LEECHBOT_DUMP_ID` | ✅ |

# Fallback inputs (if not using Secrets)
API_ID = 0  # @param {type:"integer"}
API_HASH = ""  # @param {type:"string"}
BOT_TOKEN = ""  # @param {type:"string"}
USER_ID = 0  # @param {type:"integer"}
DUMP_ID = 0  # @param {type:"integer"}

#@markdown ---
#@markdown ## ⚙️ **Deployment Options**
MOUNT_DRIVE = False  # @param {type:"boolean"}
USE_GPU = True       # @param {type:"boolean"}
ENABLE_LOGS = True   # @param {type:"boolean"}
AUTO_RESTART = True  # @param {type:"boolean"}
REPO_BRANCH = "main"  # @param ["main"]

#@markdown ---
#@markdown > 💡 **Tip**: Click **Runtime → Run all** or press **Ctrl+F9** after filling credentials.

# =============================================================================
#  📦 Imports & Setup
# =============================================================================
import subprocess, sys, os, json, time, shutil, signal
from pathlib import Path
from IPython.display import clear_output, display, Markdown
from google.colab import drive
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO if ENABLE_LOGS else logging.WARNING,
    format='[%(levelname)s] %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("LeechBot")

# =============================================================================
#  🎨 UI Components
# =============================================================================
class ColabUI:
    @staticmethod
    def banner():
        return """
╔═════════════════════════════════════════╗
║ㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤ   ║
║    🚀 LeechBot - Advanced Telegram File Transloader      ║
║ㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤ   ║
╠═════════════════════════════════════════╣
║  👤 Dev: Shinei Nouzen                ㅤㅤㅤㅤㅤㅤㅤㅤㅤ   ║
║  📂 GitHub: Shineii86/LeechBot          ㅤㅤㅤㅤㅤㅤㅤㅤㅤ ║
║  💬 Telegram: @Shineii86    ㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤㅤ ║
╚═════════════════════════════════════════╝
"""

    @staticmethod
    def status(emoji: str, msg: str, level: str = "info"):
        colors = {"info": "#2196F3", "success": "#4CAF50", "error": "#F44336", "warning": "#FF9800"}
        display(Markdown(f'<font color="{colors.get(level, "#999")}">[{emoji}] {msg}</font>'))

# =============================================================================
#  🔐 Credential Management
# =============================================================================
def get_credentials():
    creds = {}
    try:
        from google.colab import userdata
        secrets_map = {
            'API_ID': 'LEECHBOT_API_ID',
            'API_HASH': 'LEECHBOT_API_HASH',
            'BOT_TOKEN': 'LEECHBOT_BOT_TOKEN',
            'USER_ID': 'LEECHBOT_USER_ID',
            'DUMP_ID': 'LEECHBOT_DUMP_ID'
        }
        for key, secret_name in secrets_map.items():
            try:
                value = userdata.get(secret_name)
                creds[key] = int(value) if key in ['API_ID', 'USER_ID', 'DUMP_ID'] else value
                ColabUI.status("🔐", f"Loaded {key} from Colab Secrets", "success")
            except userdata.SecretNotFoundError:
                creds[key] = None
    except ImportError:
        ColabUI.status("ℹ️", "Colab Secrets not available; using manual inputs", "info")

    fallbacks = {
        'API_ID': API_ID, 'API_HASH': API_HASH, 'BOT_TOKEN': BOT_TOKEN,
        'USER_ID': USER_ID, 'DUMP_ID': DUMP_ID
    }
    for key in creds:
        if creds[key] is None:
            creds[key] = fallbacks.get(key)
    return creds

def validate_credentials(creds: dict) -> bool:
    required = ['API_ID', 'API_HASH', 'BOT_TOKEN', 'USER_ID', 'DUMP_ID']
    missing = [k for k in required if not creds.get(k)]
    if missing:
        ColabUI.status("❌", f"Missing credentials: {', '.join(missing)}", "error")
        return False
    dump_str = str(creds['DUMP_ID'])
    if len(dump_str) == 10 and not dump_str.startswith('-100'):
        creds['DUMP_ID'] = int("-100" + dump_str)
        ColabUI.status("🔄", "Auto-formatted DUMP_ID", "info")
    return True

# =============================================================================
#  🛠️ Setup Functions
# =============================================================================
def run_command(cmd: str, description: str, retries: int = 3) -> bool:
    for attempt in range(retries):
        try:
            ColabUI.status("⏳", f"{description} (attempt {attempt+1}/{retries})")
            result = subprocess.run(cmd, shell=True, capture_output=True, text=True, check=True, timeout=300)
            ColabUI.status("✅", f"{description} completed", "success")
            return True
        except subprocess.CalledProcessError as e:
            logger.warning(f"Command failed: {e.stderr[:200]}")
            if attempt == retries - 1:
                ColabUI.status("❌", f"{description} failed after {retries} attempts", "error")
                return False
            time.sleep(2 ** attempt)
        except subprocess.TimeoutExpired:
            ColabUI.status("⚠️", f"{description} timed out", "warning")
            if attempt == retries - 1:
                return False
    return False

def clone_repo(branch: str = "main") -> bool:
    repo_url = "https://github.com/Shineii86/LeechBot.git"
    target = "/content/leechbot"

    # 🔧 Fix: Ensure we are in a valid existing directory
    os.chdir("/content")

    # Clean previous install
    if os.path.exists(target):
        shutil.rmtree(target)
        ColabUI.status("🧹", "Cleaned previous installation")

    return run_command(
        f"git clone -b {branch} --depth 1 {repo_url} {target}",
        f"📥 Cloning repo (branch: {branch})"
    )

def install_dependencies() -> bool:
    if not run_command(
        "apt-get update -qq && apt-get install -y -qq ffmpeg aria2 megatools p7zip-full unzip",
        "🔧 Installing system packages"
    ):
        return False
    return run_command(
        "pip3 install -q --no-cache-dir -r /content/leechbot/requirements.txt",
        "🐍 Installing Python dependencies"
    )

def check_gpu() -> dict:
    info = {"available": False}
    if not USE_GPU:
        ColabUI.status("ℹ️", "GPU usage disabled by user", "info")
        return info
    try:
        result = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                              shell=True, capture_output=True, text=True, check=True)
        lines = result.stdout.strip().split('\n')
        if lines:
            name, memory = lines[0].split(', ')
            info.update({"available": True, "name": name.strip(), "memory": memory.strip()})
            ColabUI.status("🎮", f"GPU detected: {name} ({memory} VRAM)", "success")
    except:
        try:
            import GPUtil
            gpus = GPUtil.getAvailable(order='memory', limit=1)
            if gpus:
                info["available"] = True
                ColabUI.status("🎮", "GPU acceleration enabled", "success")
        except ImportError:
            pass
    if not info["available"]:
        ColabUI.status("ℹ️", "Using CPU fallback (no GPU detected)", "info")
    return info

def save_config(creds: dict, path: str = "/content/leechbot/credentials.json"):
    safe_creds = creds.copy()
    with open(path, 'w') as f:
        json.dump(safe_creds, f, indent=2)
    os.chmod(path, 0o600)
    ColabUI.status("💾", "Configuration saved securely", "success")

# =============================================================================
#  🚀 Main Deployment
# =============================================================================
def deploy():
    clear_output(wait=True)
    print(ColabUI.banner())

    ColabUI.status("🔐", "Loading credentials...")
    creds = get_credentials()
    if not validate_credentials(creds):
        ColabUI.status("❌", "Deployment aborted: invalid credentials", "error")
        return

    if not clone_repo(REPO_BRANCH):
        return

    if not install_dependencies():
        return

    gpu_info = check_gpu()
    save_config(creds)

    if MOUNT_DRIVE:
        try:
            drive.mount('/content/drive')
            ColabUI.status("☁️", "Google Drive mounted", "success")
        except Exception as e:
            ColabUI.status("⚠️", f"Drive mount failed: {e}", "warning")

    ColabUI.status("✨", "Finalizing setup...")

    session_files = ["/content/leechbot/my_bot.session", "/content/leechbot/my_bot.session-journal"]
    for sf in session_files:
        if os.path.exists(sf):
            os.remove(sf)

    clear_output(wait=True)
    print(ColabUI.banner())
    display(Markdown("""
### ✅ **Deployment Successful!** 🎉

| Command | Description |
|---------|-------------|
| `/start` | Initialize bot & show menu |
| `/tupload` | Upload files to Telegram |
| `/gdupload` | Mirror to Google Drive |
| `/ytupload` | Download via yt-dlp |
| `/settings` | Configure preferences |
| `/help` | Show all available commands |

> ⚠️ **Keep this tab open** while the bot runs.
"""))

    ColabUI.status("🚀", "Starting LeechBot...", "success")

    def signal_handler(sig, frame):
        ColabUI.status("🛑", "Received shutdown signal, cleaning up...")
        sys.exit(0)

    if AUTO_RESTART:
        signal.signal(signal.SIGTERM, signal_handler)
        signal.signal(signal.SIGINT, signal_handler)

    os.chdir("/content/leechbot")
    get_ipython().system('python3 -m leechbot')

if __name__ == "__main__":
    try:
        deploy()
    except KeyboardInterrupt:
        ColabUI.status("👋", "Deployment cancelled by user", "warning")
    except Exception as e:
        ColabUI.status("💥", f"Unexpected error: {e}", "error")
        if ENABLE_LOGS:
            logger.exception("Full traceback:")